In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score


In [ ]:
# ======================
# 1. ПАРАМЕТРЫ
# ======================
DATA = Path("../data/customers_rfm.csv")
FIG = Path("../figures")
FIG.mkdir(parents=True, exist_ok=True)
RANDOM_SEED = 0
K_RANGE = range(2, 8)


In [ ]:
data = pd.read_csv(DATA)


In [ ]:
print("Number of rows:", data.shape[0])
print("Columns:", list(data.columns))


In [ ]:
data.info()
data.head()


In [ ]:
data.describe().round(2)


In [ ]:
# Распределения RFM-признаков
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col, color in zip(
    axes,
    ["recency_days", "frequency", "monetary"],
    ["#440154", "#21918c", "#fde725"],
):
    sns.histplot(data[col], bins=30, ax=ax, color=color, edgecolor="white")
    ax.set_title(col, fontsize=12)
    ax.grid(axis="y", linestyle="--", alpha=0.5)

plt.suptitle("Распределения RFM-признаков", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(FIG / "rfm_distributions.png", dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
# ======================
# 2. МАСШТАБИРОВАНИЕ
# ======================
feature_cols = ["recency_days", "frequency", "monetary"]
X = data[feature_cols]
Xs = StandardScaler().fit_transform(X)
print("scaled shape:", Xs.shape)


In [ ]:
# ======================
# 3. ПОДБОР K (silhouette + inertia)
# ======================
scores = []
inertias = []
for k in K_RANGE:
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_SEED)
    labels = km.fit_predict(Xs)
    sil = silhouette_score(Xs, labels)
    scores.append((k, sil))
    inertias.append(km.inertia_)
    print(f"k={k}  silhouette={sil:.3f}  inertia={km.inertia_:.1f}")


In [ ]:
# Графики elbow и silhouette
ks = [k for k, _ in scores]
sils = [s for _, s in scores]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(ks, inertias, marker="o", color="#31688e")
axes[0].set_title("Elbow (inertia)", fontsize=13)
axes[0].set_xlabel("k")
axes[0].set_ylabel("Inertia")
axes[0].grid(True, linestyle="--", alpha=0.5)

axes[1].plot(ks, sils, marker="o", color="#35b779")
axes[1].set_title("Silhouette score", fontsize=13)
axes[1].set_xlabel("k")
axes[1].set_ylabel("Silhouette")
axes[1].grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.savefig(FIG / "rfm_k_selection.png", dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
# ======================
# 4. ФИНАЛЬНАЯ КЛАСТЕРИЗАЦИЯ
# ======================
best_k = max(scores, key=lambda t: t[1])[0]
print("best_k by silhouette:", best_k)

km_final = KMeans(n_clusters=best_k, n_init=10, random_state=RANDOM_SEED)
data = data.copy()
data["cluster"] = km_final.fit_predict(Xs)


In [ ]:
# Профили кластеров (средние RFM)
profiles = data.groupby("cluster")[feature_cols].mean().round(2)
profiles["size"] = data["cluster"].value_counts().sort_index()
profiles


In [ ]:
# ======================
# 5. ВИЗУАЛИЗАЦИЯ КЛАСТЕРОВ
# ======================
plt.figure(figsize=(9, 6))
sns.scatterplot(
    data=data,
    x="recency_days",
    y="monetary",
    hue="cluster",
    size="frequency",
    sizes=(20, 180),
    palette="viridis",
    alpha=0.7,
)
plt.title("Кластеры клиентов: Recency vs Monetary", fontsize=14, pad=15)
plt.xlabel("Recency (дни)", fontsize=12)
plt.ylabel("Monetary", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig(FIG / "rfm_scatter.png", dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
# Boxplot monetary по кластерам
plt.figure(figsize=(8, 5))
sns.boxplot(data=data, x="cluster", y="monetary", palette="viridis", hue="cluster", legend=False)
plt.title("Monetary по кластерам", fontsize=14, pad=15)
plt.xlabel("Кластер", fontsize=12)
plt.ylabel("Monetary", fontsize=12)
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig(FIG / "rfm_monetary_box.png", dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
print("Silhouette — ориентир, не доказательство. Профили кластеров смотрим бизнесом.")
print(profiles)
